# Phase 1: detector smoke test on the dev subset (Colab, **GPU runtime: T4**)
Runs D4 (CLIP B/32, L/14), D3 (FairFace classifier, UTKFace only) and D1 (MiVOLO v2 face-only) on the
500-image dev subset, then prints an **aggregate** table. Dev-subset rates are NOT results (tiny, balanced cells).

**Ethics (EXPERIMENT_PLAN §7):** never display images. Per-image outputs go to Drive `per_image/` only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, subprocess
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/child-detection-audit')
REPO = pathlib.Path('/content/child-detection-skin-tone-audit')
os.environ['CHILD_AUDIT_ENV'] = 'colab'
os.chdir('/content')  # never delete the folder we're standing in (re-runs)
!rm -rf {REPO} && git clone -q https://github.com/isaacmutuma/child-skintone-detection.git {REPO}
!git -C {REPO} log --oneline -1
os.chdir(REPO)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU: switch runtime to T4'

## 1. Dev subset → `/content/dev` (loaders pointed there via env vars)

In [ ]:
!rm -rf /content/dev && mkdir -p /content/dev && unzip -q {DRIVE_ROOT}/dev_subset.zip -d /content/dev
os.environ['UTKFACE_DIR'] = '/content/dev/UTKFace'
os.environ['FAIRFACE_DIR'] = '/content/dev/fairface'
!ls /content/dev/UTKFace | wc -l; ls /content/dev/fairface/val | wc -l

## 2. Main environment (D3, D4): open_clip; dlib should be preinstalled on Colab

In [ ]:
!pip -q install open_clip_torch huggingface_hub gdown
try:
    import dlib; print('dlib', dlib.__version__, 'cuda:', dlib.DLIB_USE_CUDA)
except ImportError:
    print('dlib missing -> installing (slow, ~10 min)'); subprocess.run(['pip', '-q', 'install', 'dlib'], check=True)

## 3. D3 face-detector files → Drive `weights/fairface` (downloaded once)
D3 = `dima806/fairface_age_image_detection` (pinned; downloaded from HF by the detector, sha256-checked).
The original FairFace ResNet-34 folder is gone (404); see PREREGISTRATION.md Deviations.

In [ ]:
W = DRIVE_ROOT / 'weights' / 'fairface'; W.mkdir(parents=True, exist_ok=True)
for f in ['mmod_human_face_detector.dat', 'shape_predictor_5_face_landmarks.dat']:
    if not (W / f).exists():
        !wget -q http://dlib.net/files/{f}.bz2 -O /content/{f}.bz2 && bunzip2 -f /content/{f}.bz2 && cp /content/{f} {W}/
!ls -la {W}

## 4. Smoke runs: D4 (both datasets), D3 (UTKFace only)

In [ ]:
runs = [('clip_vitb32', 'utkface'), ('clip_vitb32', 'fairface_val'),
        ('clip_vitl14', 'utkface'), ('clip_vitl14', 'fairface_val'),
        ('fairface_clf', 'utkface')]
for det, ds in runs:
    print(f'=== {det} / {ds}')
    !python scripts/run_detector.py --detector {det} --dataset {ds} --dev --batch-size 32

## 5. D1 (MiVOLO v2) in its own venv
MiVOLO pins old versions (torch 2.5.1, timm 0.8.13.dev0, transformers 4.51.0), so it gets an isolated env
instead of downgrading Colab's.

In [ ]:
!pip -q install uv
!rm -rf /content/venv_mivolo && uv venv -q --python 3.11 /content/venv_mivolo
PY = '/content/venv_mivolo/bin/python'
# 1) pinned deps in one resolve (torch from the CUDA 12.1 index, the rest from PyPI)
!uv pip install -q --python {PY} --index-strategy unsafe-best-match     --extra-index-url https://download.pytorch.org/whl/cu121     torch==2.5.1 torchvision==0.20.1 "numpy<2" "timm==0.8.13.dev0" "transformers==4.51.0"     "accelerate==1.8.1" "ultralytics==8.1.0" "lapx>=0.5.2" huggingface_hub pandas scipy pillow     opencv-python-headless omegaconf "setuptools<70" wheel
# 2) MiVOLO itself: its setup.py imports pkg_resources (removed from new setuptools),
#    so build without isolation against setuptools<70; deps already installed above.
!uv pip install -q --python {PY} --no-build-isolation --no-deps     "git+https://github.com/WildChlamydia/MiVOLO.git@37475e3f8818b5f22448003feec3e64b01bfb188"
!MPLBACKEND=Agg {PY} -c "import torch, transformers, ultralytics, mivolo, pandas, omegaconf; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'transformers', transformers.__version__, 'ultralytics', ultralytics.__version__)"

In [ ]:
for ds in ['utkface', 'fairface_val']:
    print(f'=== mivolo_face / {ds}')
    !MPLBACKEND=Agg {PY} scripts/run_detector.py --detector mivolo_face --dataset {ds} --dev --batch-size 16

## 6. Aggregate smoke table → Drive `results/phase1/`

In [ ]:
!python scripts/smoke_summary.py --tag dev
!mkdir -p {DRIVE_ROOT}/results/phase1 && cp -r results/phase1/* {DRIVE_ROOT}/results/phase1/
!cp results/env.json {DRIVE_ROOT}/results/env_colab.json
!ls -R {DRIVE_ROOT}/results/phase1